# Pilot Data Analysis

Analysis of pilot data from the face-flanker memory experiment. Three pilot subjects, one per between-subjects condition. See `data-guide.md` for the data dictionary and `analysis-guide.md` for full analysis rationale and interpretation guidance.

With n=1 per condition, these results verify the data pipeline and analysis code. Patterns should not be interpreted as evidence for or against the research hypotheses.

### Design recap

**Study phase (all conditions).** On each trial, a neutral target face appears flanked on both sides by the same emotional face (angry, happy, or neutral). The participant judges the target's gender. The display stays on screen for a fixed 3 seconds regardless of response. The study-phase factorial design crosses target gender (2) × flanker gender (2) × flanker emotion (3) = 12 trial types, each repeated 10 times, for 120 study trials per subject. All faces are from the Chicago Face Database; flankers are restricted to Black and White models to avoid confounding race with emotion.

**Test phase (between-subjects).** Each participant completes one of three test phases:

| Condition | Test task | Display | Trials | Key measure |
|---|---|---|---|---|
| 1 | Item recognition | Single neutral face | 240 (120 old + 120 new) | Old/new judgment |
| 2 | Associative recognition | Three faces (same layout as study) | 120 (60 intact + 60 rearranged) | Same/different judgment |
| 3 | Valence rating | Single neutral face | 120 | 1–9 valence rating |

Condition 2 rearrangement preserves all study-phase factors (target gender, flanker gender, flanker emotion) — only the specific identity pairing changes.

In [1]:
import pandas as pd
from statistics import NormalDist

z = NormalDist().inv_cdf  # equivalent to scipy.stats.norm.ppf

df = pd.read_csv('2026_02_19_pilot_3subj.csv')
df['correct'] = df['correct'].astype('boolean')  # nullable bool (handles NaN from valence trials)

print(f'{len(df)} rows, {df.subject_number.nunique()} subjects')
print(f'Conditions: {df.groupby("condition").subject_number.nunique().to_dict()}')

840 rows, 3 subjects
Conditions: {1: 1, 2: 1, 3: 1}


## Data Overview

In [2]:
overview = df.groupby(['subject_number', 'condition']).apply(
    lambda g: pd.Series({
        'study_trials': (g.phase == 'study').sum(),
        'test_trials': (g.phase == 'test').sum(),
        'total': len(g),
        'timeouts': g.timed_out.sum()
    })
).reset_index()
print(overview.to_string(index=False))

 subject_number  condition  study_trials  test_trials  total  timeouts
              1          1           120          240    360         0
              2          3           120          120    240         4
              3          2           120          120    240         1


Expected row counts: condition 1 = 360 (120 study + 240 test), conditions 2 and 3 = 240 (120 study + 120 test). All three subjects completed the expected number of trials. Timeout rates are very low (0-4 per subject), confirming the 3-second response window is adequate.

## Study Phase Analysis

The study phase is identical across conditions. Each trial displays a neutral target flanked by two copies of an emotional face; the participant judges the target's gender. The 2 × 2 × 3 factorial (target gender × flanker gender × flanker emotion) produces 12 trial types × 10 reps = 120 trials per subject.

Analyzing study-phase performance checks two things:

1. **Cover task validation** — accuracy should be near ceiling, confirming participants are attending to the faces.
2. **Flanker interference** — RT differences across flanker emotions would indicate the flankers are being processed. Because the display is shown for a fixed 3 seconds regardless of response time, RT differences do not create an encoding-time confound.

In [3]:
study = df[df.phase == 'study'].copy()

# Accuracy and RT by flanker emotion
study_by_emotion = study.groupby('flanker_emotion').apply(
    lambda g: pd.Series({
        'N': len(g),
        'accuracy': g.correct.mean(),
        'mean_rt': g.loc[~g.timed_out, 'rt'].mean(),
        'sd_rt': g.loc[~g.timed_out, 'rt'].std(),
        'timeouts': g.timed_out.sum()
    })
).round(3)
print('Study phase: accuracy and RT by flanker emotion')
print(study_by_emotion.to_string())

Study phase: accuracy and RT by flanker emotion
                     N  accuracy   mean_rt    sd_rt  timeouts
flanker_emotion                                              
angry            120.0     0.967  1232.941  399.296       2.0
happy            120.0     0.967  1159.575  391.160       0.0
neutral          120.0     1.000  1262.975  387.624       0.0


Gender judgment accuracy is near ceiling (96.7-100%), confirming the cover task works as intended. Mean RTs range from 1160 to 1263 ms — well within the 3-second window. Happy flankers produced numerically the fastest responses (1160 ms) and neutral the slowest (1263 ms), but with only 3 subjects pooled these differences are not interpretable. The 2 timeouts both occurred in the angry-flanker condition.

With the full sample, a repeated-measures ANOVA on RT with flanker emotion as the within-subjects factor would test whether emotional flankers systematically speed or slow gender judgments.

In [4]:
# Full factorial: flanker_emotion x target_gender x flanker_gender
responded = study[~study.timed_out]
factorial_rt = responded.groupby(
    ['flanker_emotion', 'target_gender', 'flanker_gender']
).agg(
    N=('rt', 'count'),
    mean_rt=('rt', 'mean'),
    sd_rt=('rt', 'std')
).round(1)
print('Study phase: mean RT by flanker emotion x target gender x flanker gender')
print(factorial_rt.to_string())

Study phase: mean RT by flanker emotion x target gender x flanker gender
                                               N  mean_rt  sd_rt
flanker_emotion target_gender flanker_gender                    
angry           female        female          30   1265.4  407.1
                              male            29   1232.2  493.1
                male          female          29   1308.7  358.3
                              male            30   1127.9  317.1
happy           female        female          30   1045.9  311.6
                              male            30   1141.0  475.5
                male          female          30   1302.9  308.9
                              male            30   1148.6  417.3
neutral         female        female          30   1073.5  280.8
                              male            30   1317.0  466.7
                male          female          30   1339.3  338.6
                              male            30   1322.1  394.2


The full 3 x 2 x 2 factorial shows substantial variability (1046-1339 ms) with ~30 observations per cell from 3 subjects pooled. No consistent pattern across target gender or flanker gender is apparent. Standard deviations (281-494 ms) are typical for speeded face-judgment tasks. This table is primarily a check that all 12 cells are populated as expected.

## RQ1: Does Emotional Context Affect Recognition Accuracy? (Condition 1)

At test, condition 1 participants see single neutral faces and judge each as old (studied) or new. The test includes all 120 studied targets plus 120 novel faces. Each studied target was originally flanked by angry, happy, or neutral faces (40 targets per emotion), but the flanker is not shown at test — the question is whether the emotional context at encoding left a trace on memory for the target itself.

**Why signal detection?** A participant who says "old" to every face achieves a perfect hit rate but has zero actual memory — they also say "old" to every new face. Raw hit rates conflate genuine memory with response tendencies. Signal detection theory (SDT) separates the two by computing **d'** (sensitivity — how well the participant can actually distinguish old from new) and **criterion c** (response bias — how willing they are to say "old" when uncertain).

**Reading d':** d' = 0 means chance (guessing). Around 0.5 is weak recognition, 1.0 is moderate, and 2.0 or above is strong. The number represents how well-separated the participant's internal "old" and "new" signals are — higher means old and new faces feel more distinct to them.

**Reading criterion c:** c = 0 is unbiased. Positive c is conservative (when uncertain, tends to say "new" — fewer hits but also fewer false alarms). Negative c is liberal (when uncertain, tends to say "old" — more hits but also more false alarms).

**Why criterion matters here:** A participant might appear to have "better memory" for emotionally-encoded faces simply because they're more willing to call those items "old" (a criterion shift) rather than actually remembering them better (a sensitivity change). Reporting both d' and c lets us distinguish these two explanations.

The false alarm rate is pooled across all new items because new items have no study history and therefore no flanker emotion assignment. This means d' differences across emotions are driven entirely by hit rate differences. If emotional flankers enhance encoding, hit rates should be higher for targets from emotional contexts, producing higher d'.

In [5]:
def edge_correct(rate, n):
    """Apply Macmillan & Kaplan (1985) edge correction."""
    if rate == 0:
        return 0.5 / n
    if rate == 1:
        return 1 - 0.5 / n
    return rate

c1_test = df[(df.condition == 1) & (df.phase == 'test')].copy()

# False alarm rate (pooled across all new items)
new_items = c1_test[c1_test.stimulus_type == 'new']
fa_rate_raw = (~new_items.correct).mean()  # said "old" to new items
fa_n = len(new_items)
fa_rate = edge_correct(fa_rate_raw, fa_n)

# Hit rate per emotion
old_items = c1_test[c1_test.stimulus_type == 'old']
rq1_rows = []
for emotion, grp in old_items.groupby('study_flanker_emotion'):
    n = len(grp)
    hit_rate_raw = grp.correct.mean()  # said "old" to old items
    hit_rate = edge_correct(hit_rate_raw, n)
    dprime = z(hit_rate) - z(fa_rate)
    criterion = -0.5 * (z(hit_rate) + z(fa_rate))
    rq1_rows.append({
        'emotion': emotion,
        'N_old': n,
        'hit_rate': round(hit_rate_raw, 3),
        'fa_rate': round(fa_rate_raw, 3),
        'd_prime': round(dprime, 3),
        'criterion': round(criterion, 3)
    })

rq1 = pd.DataFrame(rq1_rows)
print(f'RQ1: Item Recognition (n={c1_test.subject_number.nunique()} subject)')
print(f'False alarm rate (pooled): {fa_rate_raw:.3f} (N={fa_n})')
print()
print(rq1.to_string(index=False))

RQ1: Item Recognition (n=1 subject)
False alarm rate (pooled): 0.300 (N=120)

emotion  N_old  hit_rate  fa_rate  d_prime  criterion
  angry     40     0.600      0.3    0.778      0.136
  happy     40     0.625      0.3    0.843      0.103
neutral     40     0.425      0.3    0.335      0.357


**False alarm rate (.30):** This subject called 30% of never-seen faces "old." That's the baseline rate of false recognition — all three emotion conditions are compared against this same rate.

**Hit rates:** The subject correctly recognized 60% of faces studied with angry flankers and 62% with happy flankers, but only 42% of faces studied with neutral flankers. On the surface this looks like emotional context helped. But raw hit rates could reflect a greater willingness to say "old" rather than genuinely better memory — which is why we compute d'.

**d' (sensitivity):** Angry d'=0.78 and happy d'=0.84 are in the weak-to-moderate range (for reference, d'=1.0 would indicate solid recognition and 2.0 would be very strong). Neutral d'=0.34 is barely above chance. The emotional conditions show roughly twice the sensitivity of neutral — a large proportional difference, though all values are modest. This pattern is directionally consistent with the hypothesis that emotional context enhances item encoding.

**Criterion c (response bias):** The emotional conditions are nearly unbiased (c=0.14 and 0.10, close to zero), while the neutral condition is moderately conservative (c=0.36 — this subject was more reluctant to say "old" for neutral-context faces). Because the FA rate is pooled and identical across conditions, criterion differences here are a direct mathematical consequence of the hit rate differences: higher hit rate pushes c closer to zero (more liberal), lower hit rate pushes c positive (more conservative). In other words, the criterion pattern adds no new information beyond what the hit rates already show — it's d' that tells us whether actual memory sensitivity differs.

With n=1 and only 40 old items per emotion condition, these differences could easily reflect individual variation or sampling noise. The full sample would test this with a repeated-measures ANOVA on d' across flanker emotion levels, with planned comparisons of emotional (angry+happy) vs. neutral and angry vs. happy.

## RQ2: Are Items Bound More Strongly to Emotional Contexts? (Condition 2)

At test, condition 2 participants see the original three-face display (target + flankers) and judge whether the pairing is intact (same as study) or rearranged. Of the 10 reps per trial type, 5 are tested intact and 5 rearranged, giving 60 intact + 60 rearranged = 120 test trials. Rearranged trials swap flanker identities within the same trial type (preserving target gender, flanker gender, and flanker emotion), so the only cue distinguishing intact from rearranged is memory for the specific identity pairing.

This is a more demanding memory test than item recognition — it requires remembering not just the face, but who it was paired with. d' here measures associative memory strength, not just item familiarity. d' uses the same scale as RQ1 (0 = chance, ~1.0 = moderate, ~2.0 = strong), but values tend to be lower because remembering specific pairings is harder than recognizing individual faces.

Unlike RQ1, false alarm rates are computed per emotion because rearranged trials preserve the flanker emotion. Each emotion condition has its own signal distribution (intact trials) and noise distribution (rearranged trials), allowing d' to adjust for any emotion-specific response biases. There are 20 intact and 20 rearranged trials per emotion.

In [6]:
c2_test = df[(df.condition == 2) & (df.phase == 'test')].copy()

rq2_rows = []
for emotion in sorted(c2_test.flanker_emotion.unique()):
    intact = c2_test[(c2_test.flanker_emotion == emotion) & (c2_test.pair_type == 'intact')]
    rearranged = c2_test[(c2_test.flanker_emotion == emotion) & (c2_test.pair_type == 'rearranged')]
    
    n_intact = len(intact)
    n_rearranged = len(rearranged)
    hit_rate_raw = intact.correct.mean()  # said "same" to intact
    fa_rate_raw = (~rearranged.correct).mean()  # said "same" to rearranged
    
    hit_rate = edge_correct(hit_rate_raw, n_intact)
    fa_rate = edge_correct(fa_rate_raw, n_rearranged)
    
    dprime = z(hit_rate) - z(fa_rate)
    criterion = -0.5 * (z(hit_rate) + z(fa_rate))
    rq2_rows.append({
        'emotion': emotion,
        'N_intact': n_intact,
        'N_rearranged': n_rearranged,
        'hit_rate': round(hit_rate_raw, 3),
        'fa_rate': round(fa_rate_raw, 3),
        'd_prime': round(dprime, 3),
        'criterion': round(criterion, 3)
    })

rq2 = pd.DataFrame(rq2_rows)
print(f'RQ2: Associative Recognition (n={c2_test.subject_number.nunique()} subject)')
print()
print(rq2.to_string(index=False))

RQ2: Associative Recognition (n=1 subject)

emotion  N_intact  N_rearranged  hit_rate  fa_rate  d_prime  criterion
  angry        20            20      0.75     0.80   -0.167     -0.758
  happy        20            20      0.40     0.55   -0.379      0.064
neutral        20            20      0.50     0.40    0.253      0.127


**d' (sensitivity):** All three conditions are near or below zero. Angry d'=−0.17 and happy d'=−0.38 are negative, meaning this subject was actually *less* accurate than chance — they were slightly more likely to call rearranged pairings "same" than intact ones. This doesn't indicate reverse memory; with only 20 trials per cell, random variation easily pushes d' below zero when the true sensitivity is near chance. Neutral d'=0.25 is barely positive. None of these values suggest this subject could reliably distinguish intact from rearranged pairings.

**Criterion c (response bias):** The angry condition stands out with c=−0.76, a strongly liberal criterion. In concrete terms: this subject said "same" to 75% of intact trials *and* 80% of rearranged trials — they were saying "same" to almost everything regardless of whether the pairing was real. A criterion this far below zero indicates a response bias that overwhelms any memory signal. By contrast, the happy (c=0.06) and neutral (c=0.13) conditions show nearly unbiased responding — roughly equal tendencies to say "same" and "different."

Associative recognition for faces is generally harder than item recognition, so near-chance single-subject performance is not unexpected. The full sample would test whether a reliable emotion effect emerges at the group level using a repeated-measures ANOVA on d'.

## RQ3: Do Targets Acquire the Valence of Their Flankers? (Condition 3)

At test, condition 3 participants see each of the 120 studied neutral targets alone (no flanker) and rate its emotional valence on a 1–9 scale (1 = most negative, 5 = neutral, 9 = most positive). This is a direct rating measure — no signal detection is needed and there is no objectively correct answer.

If flanker emotion transfers to the target's mental representation, the predicted ordering of mean ratings is: happy > neutral > angry. Because the faces are objectively neutral, all means are expected to cluster near 5. The question is whether there are reliable between-emotion differences, even if small. Timed-out trials are excluded (no rating was recorded). There are 40 targets per emotion condition.

In [7]:
c3_test = df[(df.condition == 3) & (df.phase == 'test') & ~df.timed_out].copy()
c3_test['rating'] = pd.to_numeric(c3_test['rating'])

rq3 = c3_test.groupby('study_flanker_emotion').agg(
    N=('rating', 'count'),
    mean_rating=('rating', 'mean'),
    sd=('rating', 'std')
).round(3)

print(f'RQ3: Valence Rating (n={c3_test.subject_number.nunique()} subject)')
print(f'Timed out trials excluded: {((df.condition == 3) & (df.phase == "test") & df.timed_out).sum()}')
print()
print(rq3.to_string())

RQ3: Valence Rating (n=1 subject)
Timed out trials excluded: 2

                        N  mean_rating     sd
study_flanker_emotion                        
angry                  40        5.225  0.947
happy                  40        4.925  1.141
neutral                38        5.289  1.011


All three means cluster near the scale midpoint (4.9-5.3), as expected for objectively neutral faces. The predicted happy > neutral > angry ordering is not present — the angry condition mean (5.22) is actually slightly above happy (4.92), with neutral highest (5.29). Standard deviations are approximately 1 scale point, indicating this subject used a narrow range.

Two trials timed out and were excluded. With one subject, this is a pipeline verification: ratings are being collected, the scale is being used, and the analysis code runs correctly. The predicted pattern would be tested with a repeated-measures ANOVA on mean rating across flanker emotion levels in the full sample.

## Summary

In [8]:
print('=== Pilot Data Summary ===')
print()
print('Study phase (all subjects):')
print(f'  Overall accuracy: {study.correct.mean():.1%}')
print(f'  Mean RT: {study.loc[~study.timed_out, "rt"].mean():.0f} ms')
print()
print('RQ1 - Item Recognition (1 subject):')
for _, row in rq1.iterrows():
    print(f"  {row['emotion']:>7}: d'={row['d_prime']:.2f}, c={row['criterion']:.2f}, hit={row['hit_rate']:.2f}, FA={row['fa_rate']:.2f}")
print()
print('RQ2 - Associative Recognition (1 subject):')
for _, row in rq2.iterrows():
    print(f"  {row['emotion']:>7}: d'={row['d_prime']:.2f}, c={row['criterion']:.2f}, hit={row['hit_rate']:.2f}, FA={row['fa_rate']:.2f}")
print()
print('RQ3 - Valence Rating (1 subject):')
for emotion, row in rq3.iterrows():
    print(f'  {emotion:>7}: M={row["mean_rating"]:.2f}, SD={row["sd"]:.2f}, N={row["N"]:.0f}')

=== Pilot Data Summary ===

Study phase (all subjects):
  Overall accuracy: 97.8%
  Mean RT: 1218 ms

RQ1 - Item Recognition (1 subject):
    angry: d'=0.78, c=0.14, hit=0.60, FA=0.30
    happy: d'=0.84, c=0.10, hit=0.62, FA=0.30
  neutral: d'=0.34, c=0.36, hit=0.42, FA=0.30

RQ2 - Associative Recognition (1 subject):
    angry: d'=-0.17, c=-0.76, hit=0.75, FA=0.80
    happy: d'=-0.38, c=0.06, hit=0.40, FA=0.55
  neutral: d'=0.25, c=0.13, hit=0.50, FA=0.40

RQ3 - Valence Rating (1 subject):
    angry: M=5.22, SD=0.95, N=40
    happy: M=4.92, SD=1.14, N=40
  neutral: M=5.29, SD=1.01, N=38
